# Task 2.2 – Data Visualisation

This notebook reads detected violations from **MongoDB** and produces
two annotated analytical plots (Task 2.2.1) with interesting-point
annotations (Task 2.2.2) that support operational traffic monitoring.

| Task | Requirement | Grade target |
|------|-------------|-------------|
| 2.2.1 | Line charts – violation counts & speed patterns over arrival time | HD |
| 2.2.2 | Interesting-point annotations (max/min, spikes, percentiles) | HD |


## 1. Setup & Imports

The same `HOST_IP` used in the streaming notebook is reused here so that
the visualisation notebook can connect to the same MongoDB instance.

```
pip install matplotlib pandas pymongo numpy
```


In [ ]:
import datetime
import statistics

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pymongo import MongoClient

# Use inline backend so plots are embedded in the notebook output
%matplotlib inline

# Global style – consistent across all plots (HD requirement)
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.linestyle': '--',
    'grid.alpha': 0.4,
})

# ── Configuration ────────────────────────────────────────
HOST_IP = 'host.docker.internal'   # same value as streaming notebook
DB_NAME = 'traffic_monitoring'
COLLECTION_NAME = 'violations'
RESAMPLE_FREQ = '1min'             # bucket width for time-series aggregation
MOVING_AVG_WINDOW = 5             # matches Week-10 class example window size
SPIKE_THRESHOLD_FACTOR = 1.5      # mean × factor ⟹ spike
PERCENTILE_LEVEL = 90             # HD: dynamically computed percentile


## 2. Load Violation Data from MongoDB

Each MongoDB document has the structure:
```json
{
  'car_plate': 'ABC 123',
  'date': ISODate('2024-01-01'),
  'violations': [
    { 'violation_type': 'INSTANTANEOUS', 'timestamp_start': ...,
      'speed_reading': 115.2, ... },
    ...
  ]
}
```
The nested `violations` array is flattened into a flat DataFrame for analysis.


In [ ]:
def load_violations_from_mongo(host, port, db_name, collection_name):
    """
    Connect to MongoDB and load all violation documents.
    Flattens the nested violations array into a flat DataFrame.

    Args:
        host            : MongoDB hostname
        port            : MongoDB port (default 27017)
        db_name         : database name
        collection_name : collection name

    Returns:
        pd.DataFrame with columns:
            car_plate, date, violation_type,
            camera_id_start, camera_id_end,
            timestamp_start, timestamp_end, speed_reading
    """
    client = MongoClient(host=host, port=port)
    db = client[db_name]
    collection = db[collection_name]
    docs = list(collection.find({}, {'_id': 0}))
    client.close()

    rows = []
    for doc in docs:
        for v in doc.get('violations', []):
            rows.append({
                'car_plate':       doc['car_plate'],
                'date':            doc['date'],
                'violation_type':  v['violation_type'],
                'camera_id_start': int(v['camera_id_start']),
                'camera_id_end':   int(v['camera_id_end']),
                'timestamp_start': v['timestamp_start'],
                'timestamp_end':   v['timestamp_end'],
                'speed_reading':   float(v['speed_reading']),
            })

    if not rows:
        print('[WARNING] No violation records found in MongoDB.'
              ' Ensure the streaming notebook has been executed first.')
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df['timestamp_start'] = pd.to_datetime(df['timestamp_start'])
    df['timestamp_end']   = pd.to_datetime(df['timestamp_end'])
    df = df.sort_values('timestamp_start').reset_index(drop=True)
    return df


violations_df = load_violations_from_mongo(
    HOST_IP, 27017, DB_NAME, COLLECTION_NAME
)

print(f'[INFO] Total violation records loaded: {len(violations_df)}')
if not violations_df.empty:
    print(violations_df[['car_plate', 'violation_type',
                          'timestamp_start', 'speed_reading']].head(10))


## 3. Data Pre-processing

Violations are resampled into **1-minute buckets** to produce:

- `time_series` — per-minute violation counts split by type (INSTANTANEOUS / AVERAGE)
- `speed_series` — per-minute mean speed and a 5-window moving average
  (same rolling-window concept as the Week 10 class example)


In [ ]:
def preprocess_violations(df, resample_freq='1min', moving_avg_window=5):
    """
    Resample violation records into time-bucketed aggregations.

    Args:
        df               : flat violations DataFrame
        resample_freq    : pandas offset alias for bucket width (default '1min')
        moving_avg_window: rolling window size for speed moving average

    Returns:
        time_series  (DataFrame): per-bucket violation counts by type
        speed_series (DataFrame): per-bucket mean speed and moving average
    """
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df_indexed = df.set_index('timestamp_start').sort_index()

    # ── Violation counts per bucket ───────────────────────
    instant_counts = (
        df_indexed[df_indexed['violation_type'] == 'INSTANTANEOUS']['speed_reading']
        .resample(resample_freq).count()
        .rename('INSTANTANEOUS')
    )
    avg_counts = (
        df_indexed[df_indexed['violation_type'] == 'AVERAGE']['speed_reading']
        .resample(resample_freq).count()
        .rename('AVERAGE')
    )
    time_series = pd.concat([instant_counts, avg_counts], axis=1).fillna(0)
    time_series['TOTAL'] = time_series['INSTANTANEOUS'] + time_series['AVERAGE']

    # ── Speed stats per bucket ────────────────────────────
    speed_series = (
        df_indexed['speed_reading']
        .resample(resample_freq)
        .agg(['mean', 'max'])
        .dropna()
    )
    speed_series.columns = ['mean_speed', 'max_speed']

    # Moving average – mirrors the Week-10 statistics.mean() window approach
    speed_series['moving_avg'] = (
        speed_series['mean_speed']
        .rolling(window=moving_avg_window, min_periods=1)
        .mean()
    )

    return time_series, speed_series


time_series, speed_series = preprocess_violations(
    violations_df, RESAMPLE_FREQ, MOVING_AVG_WINDOW
)

print(f'[INFO] Time-series buckets : {len(time_series)}')
print(f'[INFO] Speed-series buckets: {len(speed_series)}')
if not time_series.empty:
    print('\nViolation count sample:')
    print(time_series.head(10))


## 4. Annotation Helper Functions

The functions below follow the same pattern as the **Week 10 Kafka Consumer
(`Scenario04`)** class example:

```python
# Week-10 class reference implementation
def annotate_max(x, y, ax=None):
    ymax = max(y)
    xpos = y.index(ymax)
    xmax = x[xpos]
    text = 'Max: Time={}, Value={}'.format(xmax, ymax)
    ax.annotate(text, xy=(xmax, ymax), xytext=(xmax, ymax+5),
                arrowprops=dict(facecolor='red', shrink=0.05))
```

Our versions extend this pattern to:
- handle **datetime x-axes** (no `.index()` on DatetimeIndex)
- skip all-zero series gracefully
- add **bbox callout boxes** for legibility
- compute **percentile thresholds** dynamically (HD requirement)
- shade **spike regions** with `axvspan` (Distinction requirement)


In [ ]:
def annotate_max(x, y, ax, color='red', label_prefix='Max'):
    """
    Annotate the maximum value on a line chart with an arrowhead callout.
    Directly based on the Week-10 Kafka Consumer (Scenario04) class example.

    Args:
        x            : list or DatetimeIndex of x-axis values
        y            : list or array-like of numeric y-values
        ax           : matplotlib Axes object
        color        : arrowhead fill colour (default 'red')
        label_prefix : text prefix shown in the annotation
    """
    y_arr = np.array(list(y), dtype=float)
    if y_arr.size == 0 or y_arr.max() == 0:
        return
    idx  = int(np.argmax(y_arr))
    xmax = list(x)[idx]
    ymax = y_arr[idx]
    y_range = float(y_arr.max() - y_arr.min()) or 1.0

    # Format timestamp the same way the class example formats it
    x_label = xmax.strftime('%H:%M') if hasattr(xmax, 'strftime') else str(xmax)
    text = f'{label_prefix}\nTime={x_label}, Value={ymax:.1f}'

    ax.annotate(
        text,
        xy=(xmax, ymax),
        xytext=(xmax, ymax + y_range * 0.28),
        arrowprops=dict(facecolor=color, edgecolor=color,
                        shrink=0.05, width=2, headwidth=8),
        fontsize=8, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='#FFFF99',
                  ec=color, alpha=0.9),
    )


def annotate_min(x, y, ax, color='orange', label_prefix='Min'):
    """
    Annotate the minimum non-zero value on a line chart with an arrowhead callout.
    Directly based on the Week-10 Kafka Consumer (Scenario04) class example.

    Args:
        x            : list or DatetimeIndex of x-axis values
        y            : list or array-like of numeric y-values
        ax           : matplotlib Axes object
        color        : arrowhead fill colour (default 'orange')
        label_prefix : text prefix shown in the annotation
    """
    y_arr  = np.array(list(y), dtype=float)
    x_list = list(x)
    # Ignore zeros so we annotate the lowest *active* period
    nonzero = [(xi, yi) for xi, yi in zip(x_list, y_arr) if yi > 0]
    if not nonzero:
        return
    xmin, ymin = min(nonzero, key=lambda t: t[1])
    y_range = float(y_arr.max() - ymin) or 1.0

    x_label = xmin.strftime('%H:%M') if hasattr(xmin, 'strftime') else str(xmin)
    text = f'{label_prefix}\nTime={x_label}, Value={ymin:.1f}'

    ax.annotate(
        text,
        xy=(xmin, ymin),
        xytext=(xmin, ymin + y_range * 0.28),
        arrowprops=dict(facecolor=color, edgecolor=color,
                        shrink=0.05, width=2, headwidth=8),
        fontsize=8, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='#FFF0CC',
                  ec=color, alpha=0.9),
    )


def annotate_percentile(x, y, ax, percentile=90, color='purple'):
    """
    Dynamically compute a percentile threshold and draw a labelled
    horizontal reference line.  Satisfies the HD requirement of Task 2.2.2:
    'Dynamically computes and labels percentiles.'

    Args:
        x          : list or DatetimeIndex of x-axis values
        y          : list or array-like of numeric y-values
        ax         : matplotlib Axes object
        percentile : integer percentile to compute (0–100), default 90
        color      : line and label colour
    """
    y_arr = np.array(list(y), dtype=float)
    if y_arr.size == 0:
        return
    p_val = float(np.percentile(y_arr, percentile))

    ax.axhline(
        y=p_val, color=color, linestyle=':', linewidth=1.8, alpha=0.85,
        label=f'P{percentile} = {p_val:.1f} km/h',
    )
    # Text label pinned to the left edge of the axis
    ax.text(
        list(x)[0], p_val * 1.005,
        f'  P{percentile} = {p_val:.1f}',
        color=color, fontsize=8, va='bottom',
    )


def detect_and_shade_spikes(x, y, ax,
                             threshold_factor=1.5,
                             color='tomato',
                             label='Spike Region'):
    """
    Detect time intervals where the value exceeds threshold_factor × mean
    and shade them with axvspan.  Satisfies the Distinction requirement of
    Task 2.2.2: 'Uses callouts or shaded regions for periods of interest.'

    Args:
        x                : list or DatetimeIndex of x-axis values
        y                : list or array-like of numeric y-values
        ax               : matplotlib Axes object
        threshold_factor : multiplier over mean to define spike threshold
        color            : shading colour
        label            : legend label for the first shaded span
    """
    y_arr  = np.array(list(y), dtype=float)
    x_list = list(x)
    mean_val = y_arr.mean()
    if mean_val == 0:
        return
    threshold = mean_val * threshold_factor
    labeled = False
    for i, yi in enumerate(y_arr):
        if yi > threshold:
            xs = x_list[max(0, i - 1)]
            xe = x_list[min(len(x_list) - 1, i + 1)]
            lbl = label if not labeled else ''
            ax.axvspan(xs, xe, alpha=0.15, color=color,
                       label=lbl, zorder=0)
            labeled = True


print('[INFO] Annotation helper functions defined.')


## 5. Plot Builder Functions

Following the Week-10 class pattern of separating
`init_plots()` → per-axis plot functions → main call:


In [ ]:
def init_plots():
    """
    Initialise a two-subplot figure for AWAS violation analysis.
    Subplot layout satisfies the Distinction requirement of Task 2.2.1:
    'Dual-axis or subplots arranged side-by-side comparison.'

    Returns:
        fig, ax1, ax2
    """
    fig = plt.figure(figsize=(14, 11))
    fig.subplots_adjust(hspace=0.55)
    ax1 = fig.add_subplot(211)   # top   – violation counts
    ax2 = fig.add_subplot(212)   # bottom – speed pattern
    fig.suptitle(
        'AWAS Traffic Monitoring – Violation Analysis Dashboard\n'
        'Violation Counts & Speed Patterns Over Arrival Time',
        fontsize=14, fontweight='bold', y=0.99,
    )
    return fig, ax1, ax2


def plot_violation_counts(ax, time_series):
    """
    Draw INSTANTANEOUS and AVERAGE violation counts over time on ax.
    Uses distinct markers and line styles (Credit requirement of Task 2.2.1).

    Args:
        ax          : matplotlib Axes
        time_series : DataFrame with DatetimeIndex and columns
                      INSTANTANEOUS, AVERAGE

    Returns:
        x      – list of datetime bucket timestamps
        y_inst – list of INSTANTANEOUS counts
        y_avg  – list of AVERAGE counts
    """
    x      = list(time_series.index)
    y_inst = list(time_series['INSTANTANEOUS'])
    y_avg  = list(time_series['AVERAGE'])

    ax.plot(x, y_inst,
            marker='o', linestyle='-',  color='crimson',
            linewidth=1.8, markersize=5,
            label='Instantaneous Violation')
    ax.plot(x, y_avg,
            marker='s', linestyle='--', color='steelblue',
            linewidth=1.8, markersize=5,
            label='Average Speed Violation')

    ax.set_xlabel('Arrival Time', fontsize=11)
    ax.set_ylabel('Violation Count', fontsize=11)
    ax.set_title('Violation Count Over Arrival Time', fontsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.tick_params(axis='x', rotation=30)

    return x, y_inst, y_avg


def plot_speed_patterns(ax, violations_df, speed_series):
    """
    Draw raw speed readings and a 5-window moving average over time on ax.
    The dual-line design mirrors the Week-10 class example (raw + moving avg).

    Args:
        ax           : matplotlib Axes
        violations_df: flat violations DataFrame
        speed_series : DataFrame with DatetimeIndex and 'moving_avg' column
    """
    vdf = violations_df.sort_values('timestamp_start')

    # Line 1 – raw individual speed readings (sparse, low alpha)
    ax.plot(
        vdf['timestamp_start'], vdf['speed_reading'],
        marker='.', linestyle='-', color='salmon',
        linewidth=0.8, markersize=3, alpha=0.5,
        label='Speed Reading (raw)',
    )
    # Line 2 – moving average (bold, class-style second line)
    ax.plot(
        speed_series.index, speed_series['moving_avg'],
        marker='^', linestyle='-', color='navy',
        linewidth=2.0, markersize=6,
        label=f'Moving Avg Speed ({MOVING_AVG_WINDOW}-window)',
    )

    ax.set_xlabel('Arrival Time', fontsize=11)
    ax.set_ylabel('Speed (km/h)', fontsize=11)
    ax.set_title('Speed Pattern Over Arrival Time', fontsize=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.tick_params(axis='x', rotation=30)


print('[INFO] Plot builder functions defined.')


## 6. Task 2.2.1 – Data Visualisation

Two subplots are produced in a single figure:

| Subplot | Content | Task 2.2.1 grade target |
|---------|---------|------------------------|
| Top (ax1) | Violation count over arrival time | Credit – title, labels, legend, distinct markers |
| Bottom (ax2) | Speed pattern + moving average | Distinction – dual-line, subplot comparison |


In [ ]:
if not violations_df.empty and not time_series.empty:
    fig, ax1, ax2 = init_plots()

    # ── Subplot 1: Violation counts ───────────────────────
    x, y_inst, y_avg = plot_violation_counts(ax1, time_series)
    ax1.legend(loc='upper right', fontsize=9)

    # ── Subplot 2: Speed pattern ──────────────────────────
    plot_speed_patterns(ax2, violations_df, speed_series)
    ax2.legend(loc='upper right', fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('task2_2_1_violation_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('[INFO] Task 2.2.1 – Visualisation saved to task2_2_1_violation_analysis.png')
else:
    print('[WARNING] No data available. Run the streaming notebook first.')


## 7. Task 2.2.2 – Interesting Point Annotation

The same dashboard is reproduced with the following annotation layers:

| Annotation | Method | Grade target |
|------------|--------|--------------|
| Peak (max) instantaneous & speed | `annotate_max()` – arrow callout (class style) | Pass |
| Low (min non-zero) instant & speed | `annotate_min()` – arrow callout (class style) | Credit |
| Spike regions shaded | `detect_and_shade_spikes()` – `axvspan` | Distinction |
| Speed-limit reference lines | `axhline` per camera | Distinction |
| 90th-percentile threshold | `annotate_percentile()` – dynamic `np.percentile` | HD |

### Operational Significance

- **Peak annotation**: identifies the highest-risk time window for patrol deployment.
- **Trough annotation**: reveals quiet periods where enforcement can be reduced.
- **Spike shading**: highlights sudden surges that may indicate an incident upstream.
- **Speed-limit lines**: instantly shows how far above the legal limit violators travel.
- **90th percentile**: separates habitual speeders (top 10%) from marginal violators,
  supporting tiered penalty enforcement.


In [ ]:
if not violations_df.empty and not time_series.empty:
    fig2, ax1, ax2 = init_plots()

    # ── Subplot 1: Violation counts + annotations ─────────
    x, y_inst, y_avg = plot_violation_counts(ax1, time_series)

    # Pass / Credit – max & min callouts (class-style arrows)
    annotate_max(x, y_inst, ax1, color='crimson',  label_prefix='Peak Instant')
    annotate_min(x, y_inst, ax1, color='orangered', label_prefix='Low Instant')
    annotate_max(x, y_avg,  ax1, color='steelblue', label_prefix='Peak Avg')

    # Distinction – shade spike regions with axvspan
    detect_and_shade_spikes(
        x, y_inst, ax1,
        threshold_factor=SPIKE_THRESHOLD_FACTOR,
        color='crimson', label='Instant Spike Region',
    )
    detect_and_shade_spikes(
        x, y_avg, ax1,
        threshold_factor=SPIKE_THRESHOLD_FACTOR,
        color='steelblue', label='Avg Spike Region',
    )

    ax1.legend(loc='upper right', fontsize=8)

    # ── Subplot 2: Speed pattern + annotations ────────────
    plot_speed_patterns(ax2, violations_df, speed_series)

    sp_x = list(speed_series.index)
    sp_y = list(speed_series['moving_avg'])

    # Pass / Credit – max & min callouts (class-style arrows)
    annotate_max(sp_x, sp_y, ax2, color='navy',    label_prefix='Peak Speed')
    annotate_min(sp_x, sp_y, ax2, color='teal',    label_prefix='Low Speed')

    # HD – dynamically computed 90th-percentile line
    annotate_percentile(
        sp_x, sp_y, ax2,
        percentile=PERCENTILE_LEVEL, color='purple',
    )

    # Distinction – speed-limit reference lines per camera
    ax2.axhline(
        y=110, color='red', linestyle='-.', linewidth=1.2, alpha=0.75,
        label='Speed Limit Cam 1 & 2 (110 km/h)',
    )
    ax2.axhline(
        y=90, color='darkorange', linestyle='-.', linewidth=1.2, alpha=0.75,
        label='Speed Limit Cam 3 (90 km/h)',
    )

    ax2.legend(loc='upper right', fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('task2_2_2_annotated_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('[INFO] Task 2.2.2 – Annotated dashboard saved to'
          ' task2_2_2_annotated_dashboard.png')
else:
    print('[WARNING] No data available. Run the streaming notebook first.')


## 8. Operational Justification

### Why these visualisations matter

**Subplot 1 – Violation count over time** reveals *when* the road is most
dangerous. Traffic enforcement units can use the peak annotation to schedule
patrols during high-risk windows, and the spike shading to detect whether a
sudden surge is isolated (incident-triggered) or sustained (systemic).

**Subplot 2 – Speed pattern with moving average** mirrors the real-time line
chart concept from the Week-10 class (raw signal + smoothed window). The
moving average filters out momentary outliers and exposes the *trend*.
Overlaying camera speed-limit lines provides immediate visual context for
how far violators exceed the legal threshold.

**90th-percentile line** separates habitual, extreme speeders from marginal
cases. Enforcement policy could apply heavier penalties to vehicles whose
speed consistently appears above this dynamically computed threshold,
without hardcoding an arbitrary cut-off.

**Spike region shading** (`axvspan`) draws the reader's eye to anomalous
time windows without requiring them to scan the entire x-axis — operationally
useful in a dashboard that may be monitored continuously during a shift.
